<a href="https://colab.research.google.com/github/zohaib-mzg/Flyrank-ML-Internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone: Refresh and Content Opportunity Scoring

This notebook mirrors the deployed paper section by section. Every number here is recomputed from the same starter CSV used since Week 2, nothing is copied in from memory.

In [1]:
import os, json, subprocess
import numpy as np
import pandas as pd

REPO_URL = "https://github.com/zohaib-mzg/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('../..')
elif not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients")

30,000 pages, 32 clients


## 1. Question

Refresh / Content Opportunity Scoring, chosen in Week 2. The question: if a content team can only review a handful of pages this week, which ones should they look at first, and why. That's a ranking and prioritization question, built on top of a classification question underneath it, is a given page currently declining.

## 2. Data

Source: the FlyRank ML Internship dataset, `data/raw/content_refresh_anonymized.csv`, a 30,000-page anonymized slice drawn from the full internship warehouse release. Grain: one row per page. Client and page identifiers are pseudonymized hashes throughout, no client names, domains, or URLs appear anywhere in this project.

In [2]:
print(f"rows: {len(df):,}")
print(f"clients: {df['client_id'].nunique()}")
print(f"median pages per client: {df.groupby('client_id').size().median():.0f}")
print(f"decline rate (trend_direction == down): {df['is_declining'].mean():.1%}")

rows: 30,000
clients: 32
median pages per client: 567
decline rate (trend_direction == down): 54.2%


Excluded deliberately: `impressions_last_30d`, `impressions_prev_30d`, `clicks_last_30d`, `clicks_prev_30d`, `sessions_last_30d`, `sessions_prev_30d`, and `trend_pct`, all of them are either the label's own source window or the label itself, caught and confirmed in the Week 3 leakage trap.

## 3. Methodology

Label: `is_declining`, `trend_direction == 'down'`, an observed comparison between two real 30-day windows already present in the export, not a rule I invented.

Features: 22 numeric and 9 categorical columns, 90-day aggregates, content metadata, and tier columns, all knowable at the decision moment. Missing values in `search_volume`, `competition`, `cpc`, `word_count`, and `char_count` filled with column medians, a real choice named rather than hidden.

Baseline: `low_ctr_visible_page`, a hand-written rule, pages with strong position, real volume, and CTR well below their tier average.

Model: Random Forest classifier, chosen over Logistic Regression because the signals genuinely interact rather than combine additively.

Validation: two passes. A plain random 75/25 split first, then a client-grouped split once the random split's leakage risk was named and checked.

## 4. Results, model vs baseline

In [3]:
numeric_features = ['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
    'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions', 'content_age_days',
    'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct']
categorical_features = ['competition_level', 'content_type', 'main_intent', 'age_tier',
    'freshness_tier', 'word_count_tier', 'char_count_tier', 'impression_tier', 'position_tier']

feature_df = df[numeric_features + categorical_features].copy()
for col in numeric_features:
    feature_df[col] = feature_df[col].fillna(feature_df[col].median())
X = pd.get_dummies(feature_df, columns=categorical_features)
y = df['is_declining']

peer_tier_avg_ctr = df.groupby('position_tier')['ctr'].transform('mean')
qualifies = (df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)
df['baseline_score'] = np.where(qualifies, (peer_tier_avg_ctr - df['ctr']) * df['impressions_90d'], -1)

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))[:k]
    return np.asarray(labels)[order].mean()

In [4]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

X_tr, X_te, y_tr, y_te, idx_tr, idx_te = train_test_split(
    X, y, df.index, test_size=0.25, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf_random.fit(X_tr, y_tr)
p50_baseline_random = precision_at_k(df.loc[idx_te, 'baseline_score'].values, y_te.values, 50)
p50_model_random = precision_at_k(rf_random.predict_proba(X_te)[:, 1], y_te.values, 50)

rf_scores, baseline_scores = [], []
for seed in range(20):
    gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_i, te_i = next(gss.split(X, y, groups=df['client_id']))
    rf_g = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
    rf_g.fit(X.iloc[tr_i], y.iloc[tr_i])
    idx_te_g = df.index[te_i]
    rf_scores.append(precision_at_k(rf_g.predict_proba(X.iloc[te_i])[:, 1], y.iloc[te_i].values, 50))
    baseline_scores.append(precision_at_k(df.loc[idx_te_g, 'baseline_score'].values, y.iloc[te_i].values, 50))

results = pd.DataFrame({
    'split': ['random (single)', 'client-grouped (20-split mean)'],
    'baseline_precision_at_50': [round(p50_baseline_random, 3), round(float(np.mean(baseline_scores)), 3)],
    'model_precision_at_50': [round(p50_model_random, 3), round(float(np.mean(rf_scores)), 3)],
    'model_std': [None, round(float(np.std(rf_scores)), 3)],
})
results

,split,baseline_precision_at_50,model_precision_at_50,model_std
0,random (single),0.520,0.880,NaN
1,client-grouped (20-split mean),0.531,0.761,0.147


The random split overstates the model's edge, 0.88 versus a client-grouped mean of 0.761 across 20 different client splits, standard deviation 0.147. Both numbers beat the baseline, but the honest one carries real, measured uncertainty the optimistic one hid.

## 5. Limitations

Only 32 clients in this slice, one with 3 pages, one with over 7,000, so any client-grouped estimate is itself unstable, hence reporting a 20-split mean and standard deviation rather than one number.

The label is a current-snapshot comparison, not a checked future outcome, `declining` means the trend looks negative right now, not that it is guaranteed to continue.

Nothing here establishes why a page declines, only that its profile resembles other pages that were observed to decline in this sample. Every claim in this project uses observed, measured, or directional language on purpose.

## 6. Ranked recommendations

In [5]:
rf_full = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf_full.fit(X, y)
df['decline_proba'] = rf_full.predict_proba(X)[:, 1]

high_conf = df['decline_proba'] >= 0.7
conditions = [high_conf & qualifies, high_conf & ~qualifies, ~high_conf & qualifies]
df['reason_code'] = np.select(conditions, ['declining_and_ctr_gap', 'declining_high_confidence', 'low_ctr_visible_page'], default='monitor_only')
df['action'] = np.select(conditions, ['refresh_and_ctr_review', 'refresh_content', 'ctr_review'], default='monitor')
df['priority_score'] = df['decline_proba'] + (qualifies).astype(float) * 0.05

queue = df.sort_values('priority_score', ascending=False).reset_index(drop=True)
queue['action'].value_counts()

,count
action,
monitor,17105
ctr_review,7411
refresh_content,3136
refresh_and_ctr_review,2348


Four actions: `refresh_and_ctr_review` (2,348 pages), `refresh_content` (3,136), `ctr_review` (7,411), `monitor` (17,105). Full reasoning and the no-go list live in Week 7's playbook notebook.

## 7. Artifacts the paper embeds

In [6]:
os.makedirs('work/outputs', exist_ok=True)

capstone_metrics = {
    'dataset': {'rows': len(df), 'clients': int(df['client_id'].nunique())},
    'random_split': {'baseline_p50': round(p50_baseline_random, 3), 'model_p50': round(p50_model_random, 3)},
    'grouped_split_20_mean': {
        'baseline_p50': round(float(np.mean(baseline_scores)), 3),
        'model_p50': round(float(np.mean(rf_scores)), 3),
        'model_std': round(float(np.std(rf_scores)), 3),
    },
    'playbook_action_counts': queue['action'].value_counts().to_dict(),
}

with open('work/outputs/capstone_metrics.json', 'w') as f:
    json.dump(capstone_metrics, f, indent=2)

print(json.dumps(capstone_metrics, indent=2))

{
  "dataset": {
    "rows": 30000,
    "clients": 32
  },
  "random_split": {
    "baseline_p50": 0.52,
    "model_p50": 0.88
  },
  "grouped_split_20_mean": {
    "baseline_p50": 0.531,
    "model_p50": 0.761,
    "model_std": 0.147
  },
  "playbook_action_counts": {
    "monitor": 17105,
    "ctr_review": 7411,
    "refresh_content": 3136,
    "refresh_and_ctr_review": 2348
  }
}
